# Day 4: Printing Department

[*Advent of Code 2025 day 4*](https://adventofcode.com/2025/day/4) and [*solution megathread*](https://redd.it/1pdr8x6)

[![nbviewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.jupyter.org/github/UncleCJ/advent-of-code/blob/cj/2025/04/code.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/UncleCJ/advent-of-code/cj?filepath=2025%2F04%2Fcode.ipynb)

In [1]:
from IPython.display import HTML
import sys
sys.path.append('../../')


# %load_ext nb_mypy
# %nb_mypy On

In [2]:
import common


downloaded = common.refresh()
%store downloaded >downloaded

# %load_ext pycodestyle_magic
# %pycodestyle_on

Writing 'downloaded' (dict) to file 'downloaded'.


In [3]:
from IPython.display import HTML

HTML(downloaded['part1'])

In [4]:
part1_example_input = '''..@@.@@@@.
@@@.@.@.@@
@@@@@.@.@@
@.@@@@..@.
@@.@@@@.@@
.@@@@@@@.@
.@.@.@.@@@
@.@@@.@@@@
.@@@@@@@@.
@.@.@@@.@.'''

inputdata = downloaded['input']

In [5]:
def parse_input(lines: Iterable[str]) -> list[list[bool]]:
    parsed_input = []
    for line in lines:
        parsed_input.append(list(c == '@' for c in line))
    return parsed_input

parsed_input = parse_input(part1_example_input.splitlines())
# parsed_input = parse_input(inputdata.splitlines())

In [6]:
print(parsed_input)

[[False, False, True, True, False, True, True, True, True, False], [True, True, True, False, True, False, True, False, True, True], [True, True, True, True, True, False, True, False, True, True], [True, False, True, True, True, True, False, False, True, False], [True, True, False, True, True, True, True, False, True, True], [False, True, True, True, True, True, True, True, False, True], [False, True, False, True, False, True, False, True, True, True], [True, False, True, True, True, False, True, True, True, True], [False, True, True, True, True, True, True, True, True, False], [True, False, True, False, True, True, True, False, True, False]]


In [7]:
def format_matrix(matrix: Iterable[Iterable[bool | int]]) -> str:
    lines = []
    for line in matrix:
        if isinstance(line[0], bool):
            lines.append(''.join(list('@' if c == True else '.' for c in line)))
        elif isinstance(line[0], int):
            lines.append(''.join(list(str(c) if c > 0 else '.' for c in line)))
    return '\n'.join(lines)

In [8]:
print(format_matrix(parsed_input))

..@@.@@@@.
@@@.@.@.@@
@@@@@.@.@@
@.@@@@..@.
@@.@@@@.@@
.@@@@@@@.@
.@.@.@.@@@
@.@@@.@@@@
.@@@@@@@@.
@.@.@@@.@.


In [9]:
def generate_adjacent(row: int, col: int, row_count: int, col_count: int) -> Iterator[tuple[int, int]]:
    for adj_row, adj_col in ((row + row_dir, col + col_dir) for row_dir, col_dir in 
                             ((-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1))):
        if (adj_row < 0 or adj_col < 0 or
                adj_row > row_count - 1 or adj_col > col_count - 1):
            continue
        yield (adj_row, adj_col)

In [10]:
def count_adjacent_pos(row: int, col: int, floorplan: list[list[bool]]) -> int:
    row_count = len(floorplan)
    col_count = len(floorplan[0])
    result = 0
    for adj_row, adj_col in generate_adjacent(row, col, row_count, col_count):
        result += floorplan[adj_row][adj_col]
    return result

In [11]:
def count_adjacent(matrix: list[list[bool]]) -> list[list[int]]:
    row_count = len(matrix)
    col_count = len(matrix[0])
    results = list([0]*col_count for _ in range(row_count))
    for row in range(row_count):
        for col in range(col_count):
            if matrix[row][col]:  # If position contains a roll, count its neighbors
                results[row][col] = count_adjacent_pos(row, col, matrix)
    return results

In [12]:
print(format_matrix(count_adjacent(parsed_input)))

..33.3343.
366.4.4.54
47675.2.44
4.6776..4.
35.7875.43
.4657665.4
.4.6.5.674
2.666.6774
.55767675.
1.3.454.2.


In [13]:
def count_accessible(floorplan: list[list[bool]]) -> tuple[int, list[list[int]]]:
    accessible_count = 0
    adj_count = count_adjacent(floorplan)
    row_count = len(floorplan)
    col_count = len(floorplan[0])
    for row in range(row_count):
        for col in range(col_count):
            if floorplan[row][col]:
                accessible_count += adj_count[row][col] < 4
    return (accessible_count, adj_count)

In [14]:
print(count_accessible(parsed_input)[0])

13


In [15]:
HTML(downloaded['part1_footer'])

In [16]:
HTML(downloaded['part2'])

In [17]:
def remove_pos(row: int,
               col: int,
               floorplan: list[list[bool]],
               adj_count: list[list[int]]) -> tuple[list[list[bool]], list[list[int]]]:
    assert(adj_count[row][col] < 4)
    row_count = len(floorplan)
    col_count = len(floorplan[0])
    for adj_row, adj_col in generate_adjacent(row, col, row_count, col_count):
        if adj_count[adj_row][adj_col] > 0:
            adj_count[adj_row][adj_col] -= 1
        adj_count[row][col] = 0
        floorplan[row][col] = False
    return floorplan, adj_count

In [18]:
def remove_accessible_pass(floorplan: list[list[bool]],
                           adj_count: list[list[int]]) -> tuple[int, list[list[bool]], list[list[int]]]:
    removed_count = 0
    row_count = len(floorplan)
    col_count = len(floorplan[0])
    for row in range(row_count):
        for col in range(col_count):
            if floorplan[row][col] and adj_count[row][col] < 4:
                adjacent_pos = count_adjacent_pos(row, col, floorplan)
                remove_pos(row, col, floorplan, adj_count)
                removed_count += 1
    return (removed_count, floorplan, adj_count)

In [19]:
def remove_accessible(floorplan: list[list[bool]]) -> int:
    total_removed = 0
    accessible, adj_count = count_accessible(floorplan)
    while accessible > 0:
        accessible, floorplan, adj_count = remove_accessible_pass(floorplan, adj_count)
        total_removed += accessible
    return total_removed

In [20]:
print(remove_accessible(parsed_input))

43


In [21]:
HTML(downloaded['part2_footer'])